# 🔬 PaliGemma Model Evaluation - UFV Campus Dataset

**TFG: VisionLeaderUFV**  
**Author:** Alberto Gómez Soteres  
**Universidad Francisco de Vitoria**

---

This notebook evaluates PaliGemma models on the **UFV Campus dataset** with multiple question types.

## Experiment Setup:
- **Dataset**: UFV Campus only
- **Questions**: Binary (yes/no, up/down, open/closed), Element identification, and Captions
- **Expected Answers**: Per-image specific (loaded from `ufv_expected_answers.json`)

## Metrics Evaluated:
- **Exact Match**: For binary questions
- **Keyword F1**: For element identification and captions
- **BERTScore**: For caption semantic similarity
- **Inference Time**: Performance metric
- **VRAM Usage**: GPU memory consumption

## 1️⃣ Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers>=4.40.0 accelerate bitsandbytes
!pip install -q torch torchvision
!pip install -q pandas numpy tqdm Pillow

print("✅ All packages installed successfully!")

In [ ]:
# Mount Google Drive (for data only)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted (for data access)!")

In [ ]:
# Setup project - Clone from GitHub (Private Repo)
import os
import sys
import subprocess

# ============================================================
# CONFIGURATION - PRIVATE REPO
# ============================================================
GITHUB_USER = "AlbertoGomezS"
GITHUB_REPO = "tfg-AlbertoGomez"
LOCAL_PROJECT_PATH = '/content/tfg'

# GitHub Personal Access Token (create at: https://github.com/settings/tokens)
# Required scopes: repo (full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # <-- REPLACE THIS

# ============================================================

# Clone or update from GitHub
if os.path.exists(LOCAL_PROJECT_PATH):
    # Update existing repo
    os.chdir(LOCAL_PROJECT_PATH)
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode == 0:
        print("✅ Repository updated from GitHub")
    else:
        print(f"⚠️ Git pull issue: {result.stderr}")
else:
    # Clone with authentication (private repo)
    clone_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    result = subprocess.run(['git', 'clone', clone_url, LOCAL_PROJECT_PATH], capture_output=True, text=True)
    
    if result.returncode == 0:
        os.chdir(LOCAL_PROJECT_PATH)
        print(f"✅ Repository cloned from GitHub (private)")
    else:
        print(f"❌ Clone failed. Check your GITHUB_TOKEN")
        print(f"   Error: {result.stderr}")
        raise Exception("Could not clone repository - check token")

# Add src to path
sys.path.insert(0, LOCAL_PROJECT_PATH)

print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# Import project modules
from src.model_loader import load_paligemma, unload_model, print_gpu_memory, get_model_info
from src.dataset_loader import (
    load_questions_config,
    load_dataset,
    create_evaluation_items,
    print_dataset_summary,
    load_per_image_answers
)
from src.inference import (
    run_inference_on_dataset,
    print_inference_summary,
    results_to_dict
)
from src.metrics import (
    calculate_all_metrics,
    print_metrics_summary,
    get_gpu_memory_usage,
    reset_gpu_memory_stats
)

import json
from datetime import datetime

print("✅ All modules imported successfully!")

## 2️⃣ Configuration

In [ ]:
# ============================================================
# CONFIGURATION - PATHS
# ============================================================

# Data is stored in Google Drive (too large for Git)
DATA_PATH = '/content/drive/MyDrive/tfg/data/UFV Campus'  # Only UFV Campus dataset

# Config and src are cloned from GitHub
CONFIG_PATH = '/content/tfg/config'

# Results stored in Drive for persistence
RESULTS_PATH = '/content/drive/MyDrive/tfg/results/ufv_experiment'

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verify data path exists
if not os.path.exists(DATA_PATH):
    print(f"⚠️ Data path not found: {DATA_PATH}")
    print("   Please upload your UFV Campus data folder to Google Drive")
else:
    print(f"✅ Data path found: {DATA_PATH}")
    
print(f"📁 Config path: {CONFIG_PATH}")
print(f"📁 Results path: {RESULTS_PATH}")

In [ ]:
# ============================================================
# SELECT MODEL TO EVALUATE
# ============================================================

# Choose ONE model to evaluate (for initial testing)
MODEL_TO_EVALUATE = "google/paligemma2-3b-pt-448"  # Change this to test different models

# Alternative models to try:
# "google/paligemma-3b-mix-224"
# "google/paligemma2-3b-pt-448"

print(f"🧪 Evaluating model: {MODEL_TO_EVALUATE}")

In [ ]:
# Load configurations
questions_config = load_questions_config(f"{CONFIG_PATH}/questions.json")

# Load per-image expected answers
per_image_answers = load_per_image_answers(f"{CONFIG_PATH}/ufv_expected_answers.json")

print(f"✅ Loaded {len(questions_config['categories'])} question categories")
print(f"✅ Loaded expected answers for {len(per_image_answers)} images")

In [ ]:
# Load UFV Campus dataset
samples = load_dataset(DATA_PATH, questions_config, load_images=False)

# Print dataset summary
print_dataset_summary(samples)

In [ ]:
# Create evaluation items (image + question pairs)
# Pass per_image_answers to override default expected answers
eval_items = create_evaluation_items(samples, questions_config, per_image_answers=per_image_answers)

print(f"\n📊 Total evaluation items: {len(eval_items)}")
print(f"   (Each image has multiple questions depending on category)")

In [ ]:
# Preview some evaluation items
print("\n📋 Sample evaluation items:")
print("-" * 60)
for i, item in enumerate(eval_items[:6]):
    print(f"\n{i+1}. Image: {item.image_sample.image_id}")
    print(f"   Category: {item.image_sample.category}")
    print(f"   Question: {item.question.question_text}")
    print(f"   Type: {item.question.question_type}")
    if item.question.expected_answer:
        print(f"   Expected: {item.question.expected_answer}")

## 4️⃣ HuggingFace Login

In [ ]:
# Login to HuggingFace (required for PaliGemma access)
from huggingface_hub import login

# IMPORTANT: Replace with your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
# Make sure you have accepted the PaliGemma license

HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"  # <-- REPLACE THIS

if HF_TOKEN != "YOUR_HUGGINGFACE_TOKEN":
    login(token=HF_TOKEN)
    print("✅ Logged in to HuggingFace")
else:
    print("⚠️ Please set your HuggingFace token above")
    print("   1. Get your token from: https://huggingface.co/settings/tokens")
    print("   2. Accept the PaliGemma license")
    print("   3. Replace 'YOUR_HUGGINGFACE_TOKEN' with your actual token")

## 5️⃣ Run Evaluation

In [ ]:
print(f"🚀 Starting evaluation of model: {MODEL_TO_EVALUATE}")
print(f"📊 Total items to evaluate: {len(eval_items)}")
print("=" * 60)

In [ ]:
# Get model info
model_info = get_model_info(MODEL_TO_EVALUATE)
print(f"📦 Model: {MODEL_TO_EVALUATE}")
print(f"   Version: {model_info['version']}, Size: {model_info['size']}, "
      f"Type: {model_info['type']}, Resolution: {model_info['resolution']}")

# Reset GPU memory stats
reset_gpu_memory_stats()

# Load model
print("\n🔄 Loading model...")
try:
    model, processor = load_paligemma(MODEL_TO_EVALUATE)
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

# Run inference
print("\n🔬 Running inference...")
results = run_inference_on_dataset(
    model=model,
    processor=processor,
    eval_items=eval_items,
    max_new_tokens=250,  # Longer responses needed for captions
    show_progress=True
)

# Get VRAM usage after inference
vram_usage = get_gpu_memory_usage()

# Print inference summary
print_inference_summary(results)

# Convert results to dict format
results_dict = results_to_dict(results)

# Calculate metrics
print("\n📊 Calculating metrics...")
metrics = calculate_all_metrics(
    results_dict,
    calculate_bertscore_metric=True,  # Needed for caption questions
    vram_usage=vram_usage
)

# Print metrics summary
print_metrics_summary(metrics)

# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name_safe = MODEL_TO_EVALUATE.replace("/", "_").replace("-", "_")

# Save raw results
results_file = f"{RESULTS_PATH}/{model_name_safe}_results_{timestamp}.json"
with open(results_file, 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f"\n💾 Results saved to: {results_file}")

# Save metrics
metrics_file = f"{RESULTS_PATH}/{model_name_safe}_metrics_{timestamp}.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"💾 Metrics saved to: {metrics_file}")

# Unload model to free memory
print("\n🧹 Unloading model...")
unload_model(model, processor)
print_gpu_memory()

print("\n" + "=" * 60)
print("✅ EVALUATION COMPLETE!")
print("=" * 60)

## 6️⃣ Results Summary

In [ ]:
# Display overall metrics
overall = metrics.get('overall', {})
print("\n📊 OVERALL METRICS:")
print("=" * 60)
print(f"Exact Match:        {overall.get('exact_match', 0):.4f}")
print(f"Keyword F1:         {overall.get('keyword_f1', 0):.4f}")
if 'bertscore_f1' in overall:
    print(f"BERTScore F1:       {overall.get('bertscore_f1', 0):.4f}")
print(f"Avg Inference Time: {overall.get('avg_inference_time', 0):.2f} seconds")
print(f"VRAM Usage:         {overall.get('vram_max_allocated_gb', 0):.2f} GB")
print("=" * 60)

# Display metrics by category
print("\n📊 METRICS BY CATEGORY:")
print("=" * 60)
for category, cat_metrics in metrics.items():
    if category == 'overall':
        continue
    print(f"\n{category.upper()}:")
    print(f"  Exact Match: {cat_metrics.get('exact_match', 0):.4f}")
    print(f"  Keyword F1:  {cat_metrics.get('keyword_f1', 0):.4f}")
    if 'bertscore_f1' in cat_metrics:
        print(f"  BERTScore F1: {cat_metrics.get('bertscore_f1', 0):.4f}")
print("=" * 60)